# RunPod: expert, checkpoints, dataset, D0, all on the Hugging Face Hub

Run the cells from the top, on an A100 pod with the network volume at `/workspace` and this repository cloned to `/workspace/AAC`.

| step | what |
|---|---|
| 0 | configuration, helpers, Hugging Face login (a wrong token fails now, not after hours), environment |
| 1 | **smoke test** of the expert training: seconds per update and hours for 1e8 to 1e9 steps, and the real command on a few updates |
| 2 | **expert training**, detached (survives a closed tab; run the cell again after a pod restart and it resumes), then the checkpoints to the Hub |
| 3 | choose the early snapshots, **collect the dataset** (train, val, test), detached |
| 4 | replay check on the CPU, **D0**, data and D0 report to the Hub |

The Hub repo is one *dataset* repo: `checkpoints/`, `data/`, `reports/`. A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
from pathlib import Path

VOLUME = Path("/workspace")
REPO = VOLUME / "AAC"                      # the git clone
VENV = VOLUME / "venvs" / "jax"            # JAX environment, on the volume: a restarted pod reuses it
DATA = VOLUME / "data"                     # episode files, reports, logs (HJEPA_HOME)
MODELS = VOLUME / "models"                 # expert checkpoints

HF_REPO_ID = "your-user/craftax-classic"   # dataset repo for checkpoints, data and reports (created if missing)
HF_PRIVATE = True

TOTAL_STEPS = 1_000_000_000                # expert run length; it also sets the learning-rate and epsilon schedules, so it cannot be extended later
EXPERT_OVERRIDES = []                      # KEY=value of the expert's configuration, e.g. ["NORM_TYPE=batch_norm", "GAMMA=0.995"]

# the dataset: frames per split and the share of each source (placeholders, see README: Data)
FRAMES = {"train": 2_000_000, "val": 150_000, "test": 150_000}
SHARE = {"train": {"expert": 0.7, "early": 0.3}, "val": {"expert": 0.7, "early": 0.3}, "test": {"expert": 1.0}}
REPLAY_EPISODES = 10                       # episodes per file replayed on the CPU in step 4

In [ ]:
# --- 0. Helpers, Hugging Face login ----------------------------------------------------------------------------------
import getpass, hashlib, json, os, shutil, subprocess, sys, time

ENV_NAME = "Craftax-Classic-Symbolic-v1"
RUN = f"pqn_rnn_{ENV_NAME}_seed0"
CKPT_DIR = MODELS / ENV_NAME
FINAL = CKPT_DIR / f"{RUN}_vmap0.safetensors"
STATE = CKPT_DIR / f"{RUN}_state.safetensors"
PYTHON = VENV / "bin" / "python"
LOGS = DATA / "logs"
ENV = {
    # LD_LIBRARY_PATH is dropped: the RunPod image puts its own CUDA 12.8 there, JAX then loads that cuBLAS next to the
    # pip-installed CUDA 12.9 libraries and segfaults at the first matrix multiply
    **{k: v for k, v in os.environ.items() if k != "LD_LIBRARY_PATH"},
    "PATH": f"{VENV / 'bin'}:{os.environ['PATH']}",   # `python` in the runner's commands is the venv's
    "HJEPA_HOME": str(DATA),
    "HDF5_USE_FILE_LOCKING": "FALSE",                 # HDF5 can fail to lock files on a network volume
    "PYTHONUNBUFFERED": "1",
    "PYTHONFAULTHANDLER": "1",                        # a crash in a subprocess prints the Python stack
}
LOGS.mkdir(parents=True, exist_ok=True)


def sh(cmd, cwd=None, env=None, check=True, quiet=False):
    """Run a command and stream its output into the cell; `quiet` shows the last lines only if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join((c.splitlines() or [""])[0][:100] + (" ..." if len(c.splitlines()) > 1 or len(c) > 100 else "") for c in cmd))
    try:
        proc = subprocess.Popen(cmd, env=env or ENV, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except FileNotFoundError:
        if check:
            raise
        print(f"  ({cmd[0]} not found)")
        return 127
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not quiet:
            print(line, end="")
    code = proc.wait()
    if code and quiet:
        print("".join(lines[-30:]), end="")
    if check and code:
        raise RuntimeError(f"{cmd[0]} exited with {code}")
    return code


PROCS = {}


def launch(name, cmd, cwd):
    """Start a command detached from the kernel (new session), log to LOGS/<name>.log, pid in LOGS/<name>.pid."""
    if running(name):
        return print(f"{name} is already running (pid {(LOGS / (name + '.pid')).read_text()})")
    with open(LOGS / f"{name}.log", "a") as log:
        PROCS[name] = subprocess.Popen([str(c) for c in cmd], env=ENV, cwd=cwd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    (LOGS / f"{name}.pid").write_text(str(PROCS[name].pid))
    print(f"started {name}, pid {PROCS[name].pid}, log {LOGS / (name + '.log')}")


def running(name):
    if name in PROCS:                      # poll() also reaps a finished child (else it stays a zombie and looks alive)
        return PROCS[name].poll() is None
    try:                                   # after a kernel restart the process is no longer our child
        os.kill(int((LOGS / f"{name}.pid").read_text()), 0)
        return True
    except (FileNotFoundError, ValueError, ProcessLookupError):
        return False


def progress(name, key, n=1):
    """The last `n` log lines that contain `key` (the progress lines; the rest of the log is start-up noise)."""
    path = LOGS / f"{name}.log"
    lines = [l for l in path.read_text().splitlines() if key in l] if path.exists() else []
    return [l[:200] for l in lines[-n:]] or ["(no progress line yet)"]


def wait(name, key, every=60):
    while running(name):
        time.sleep(every)
        print(time.strftime("%H:%M"), progress(name, key)[0])
    print(f"{name} finished")


# the Hugging Face login: a wrong token or repo id fails here, before anything long runs
sh([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=2"], env=os.environ)
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (write access): "))
print("logged in as", api.whoami()["name"])
api.create_repo(HF_REPO_ID, repo_type="dataset", private=HF_PRIVATE, exist_ok=True)

In [ ]:
# --- 0. The pod and the JAX GPU environment (installed once) --------------------------------------------------------
assert REPO.exists(), f"clone the repository to {REPO}"
for command in (["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], ["nproc"], ["df", "-h", str(VOLUME)]):
    sh(command, env=os.environ, check=False)
sh(["git", "-C", REPO, "log", "-1", "--oneline"], env=os.environ)

files = [REPO / "environment" / "jax-gpu-requirements.txt", REPO / "environment" / "jax-requirements.txt", REPO / "purejaxql" / "pyproject.toml"]
stamp = hashlib.sha256(b"".join(f.read_bytes() for f in files)).hexdigest()[:16]
marker = VENV / ".installed"
if not (marker.exists() and marker.read_text() == stamp):
    if not PYTHON.exists():
        sh([sys.executable, "-m", "venv", VENV], env=os.environ)
    sh([PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
    sh([PYTHON, "-m", "pip", "install", "-q", "-r", "environment/jax-gpu-requirements.txt"], cwd=REPO)
    marker.write_text(stamp)
platform = subprocess.run([PYTHON, "-c", "import jax; print(jax.devices()[0].platform)"], env=ENV, capture_output=True, text=True).stdout.strip()
assert platform == "gpu", "JAX does not see the GPU: check nvidia-smi and the jax[cuda12] install in the venv"
print("JAX sees the GPU")

## 1. Smoke test of the expert training

First `time_expert.py`: seconds per update, steps per second, the cost of one greedy test and the hours for 1e8 to 1e9 steps (read the 1e9 row against your budget; the time of the snapshot writes is not in it). Then the **real training command** on 6 updates into `/workspace/smoke`: this proves the command, the paths and the checkpoint writes on this pod before the long run.

In [ ]:
# --- 1. Smoke test -----------------------------------------------------------------------------------------------------
def train_command(total, save_path, extra=()):
    """The expert training command. Every override needs the `alg.` prefix: a top-level KEY=value is overridden by the file's own."""
    overrides = dict(o.split("=", 1) for o in [*EXPERT_OVERRIDES, *extra])
    return [PYTHON, "purejaxql/pqn_rnn_craftax.py", "+alg=pqn_rnn_craftax_classic", f"alg.TOTAL_TIMESTEPS={total}",
            f"alg.TOTAL_TIMESTEPS_DECAY={total}", f"alg.SAVE_PATH={save_path}", *[f"alg.{k}={v}" for k, v in overrides.items()]]


sh([PYTHON, "scripts/time_expert.py", *EXPERT_OVERRIDES], cwd=REPO / "purejaxql")

print("")
steps_code = "import sys; sys.path.insert(0, 'scripts'); import time_expert as t; c = t.load_config(sys.argv[1:]); print(c['NUM_ENVS'] * c['NUM_STEPS'])"
steps_per_update = int(subprocess.run([PYTHON, "-c", steps_code, *EXPERT_OVERRIDES], env=ENV, cwd=REPO / "purejaxql", capture_output=True, text=True).stdout.split()[-1])
SMOKE = VOLUME / "smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
t0 = time.time()
sh(train_command(6 * steps_per_update, SMOKE, ["CHECKPOINT_INTERVAL=0.5", "TEST_INTERVAL=0.5"]), cwd=REPO / "purejaxql", quiet=True)
smoke = SMOKE / ENV_NAME
assert (smoke / FINAL.name).exists() and list(smoke.glob("*_u0000003.json")), "the smoke run did not write its checkpoints"
print(f"smoke run ok in {time.time() - t0:.0f} s (6 updates, 2 chunks, 2 greedy tests, compilation included); state file {(smoke / STATE.name).stat().st_size / 2**20:.0f} MiB")

## 2. Expert training (detached)

Set `TOTAL_STEPS` (and `EXPERT_OVERRIDES`) in the configuration cell from the smoke test, then start. The run prints a line every 5% and writes a snapshot with its greedy-test metrics at the same points. If the pod restarts, run the cell again: it resumes from the saved state with the same configuration.

In [ ]:
# --- 2. Start (or resume) the expert training, detached ---------------------------------------------------------------------
if FINAL.exists():
    print("the expert is already trained:", FINAL)
else:
    launch("expert", train_command(TOTAL_STEPS, MODELS, ["RESUME=true"] if STATE.exists() else []), REPO / "purejaxql")

In [ ]:
# --- 2. Monitor the expert (run as often as you like) -------------------------------------------------------------------------
def snapshot_table():
    """The snapshots with their greedy-test metrics: choose the early snapshots for the dataset from this."""
    print("update | steps (M) | return | length | achievements >50% | iron pickaxe | diamond")
    for f in sorted(CKPT_DIR.glob("*_vmap0_u*.json")):
        d = json.loads(f.read_text())
        t = d.get("test", {})
        ach = {k.split("/", 1)[1]: v for k, v in t.items() if k.startswith("Achievements/")}
        print(f"{d['n_updates']:6d} | {d['env_steps'] / 1e6:9.0f} | {t.get('returned_episode_returns', float('nan')):6.1f} | {t.get('returned_episode_lengths', float('nan')):6.0f} | "
              f"{sum(v > 0.5 for v in ach.values()):17d} | {ach.get('make_iron_pickaxe', 0):12.2f} | {ach.get('collect_diamond', 0):.3f}")


print("expert", "RUNNING" if running("expert") else "not running", "| final policy", "exists" if FINAL.exists() else "not yet")
print("\n".join(progress("expert", "update ", 3)))
snapshot_table()

In [ ]:
# --- 2. Wait for the expert to finish, then upload the checkpoints ------------------------------------------------------------
wait("expert", "update ")
assert FINAL.exists(), "the run ended without a final policy: read the log, then run the start cell again to resume"
snapshot_table()
# the policies and their greedy-test metrics (not the resumable state, which is as large as the stored transitions)
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(CKPT_DIR), path_in_repo="checkpoints", ignore_patterns=["*_state.safetensors"])
print("checkpoints uploaded to", f"https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/checkpoints")

## 3. The dataset

Choose the **early snapshots** from the table above: policies at different stages of the chain that still fail (die in fights, starve). Their rollouts are the early share of the data. The final policy is the expert. The collection runs the three splits one after the other; a restarted run skips the finished steps.

In [ ]:
# --- 3. Choose the early snapshots and write the plan --------------------------------------------------------------------------
EARLY_UPDATES = []        # training updates of the chosen snapshots, from the table, e.g. [381, 1143, 2286]
assert EARLY_UPDATES, "set EARLY_UPDATES from the snapshot table"
EARLY = [CKPT_DIR / f"{RUN}_vmap0_u{u:07d}.safetensors" for u in EARLY_UPDATES]
assert FINAL.exists() and all(p.exists() for p in EARLY), "a checkpoint is missing"

# a copy of the experiment spec with the frames and shares of the configuration cell
SPEC = DATA / "spec.yaml"
spec_code = """
import json, sys, yaml
from pathlib import Path
repo, out, frames, share = sys.argv[1:5]
spec = yaml.safe_load(Path(repo, "experiments", "experiments.yaml").read_text())
spec["collection"]["frames"], spec["collection"]["share"] = json.loads(frames), json.loads(share)
Path(out).write_text(yaml.safe_dump(spec, sort_keys=False))
"""
sh([PYTHON, "-c", spec_code, REPO, SPEC, json.dumps(FRAMES), json.dumps(SHARE)])
RUNNER = [PYTHON, REPO / "experiments" / "run.py"]
D1 = [*RUNNER, "run", "D1", "--spec", SPEC, "--mode", "local", "--expert-checkpoint", FINAL, *[x for p in EARLY for x in ("--early-checkpoint", p)]]
sh([*RUNNER, "plan", "D1", "--spec", SPEC, "--expert-checkpoint", FINAL, *[x for p in EARLY for x in ("--early-checkpoint", p)]])

In [ ]:
# --- 3. Collect, detached ------------------------------------------------------------------------------------------------------
launch("collect", D1, REPO)

In [ ]:
# --- 3. Monitor the collection (run as often as you like) ------------------------------------------------------------------------
STATUS = """
import json, sys, h5py, numpy as np
home, frames = sys.argv[1], json.loads(sys.argv[2])
sources = {0: "expert", 1: "noised", 2: "random", 3: "early"}
for split in ("train", "val", "test"):
    try:
        f = h5py.File(f"{home}/craftax_expert_{split}.h5", "r")
    except OSError:
        print(f"{split:5s} no file yet"); continue
    with f:
        lengths, offsets = f["ep_len"][:], f["ep_offset"][:]
        source = f["source"][offsets] if len(offsets) else np.zeros(0)
        by = ", ".join(f"{n} {int(lengths[source == c].sum()):,}" for c, n in sources.items() if (source == c).any())
        print(f"{split:5s} {len(lengths):6d} episodes {int(lengths.sum()):11,d} frames of {frames[split]:,} ({by})")
"""
print("collection", "RUNNING" if running("collect") else "not running", "| finished runs:", sorted(p.name for p in (DATA / "collect_done").glob("*")))
sh([PYTHON, "-c", STATUS, DATA, json.dumps(FRAMES)], check=False)
print("\n".join(progress("collect", "tried ", 3)))

In [ ]:
# --- 3. Wait for the collection to finish ------------------------------------------------------------------------------------------
wait("collect", "tried ")
assert "failed with exit code" not in (LOGS / "collect.log").read_text(), "a collection run failed: read the log, run the start cell again"

## 4. Checks, D0 and the upload

**Replay:** the planning environment runs on the CPU and the episodes were made on the GPU, so a sample of every file is replayed on the CPU and compared column by column; the discrete columns must be identical. **D0:** the dataset analysis; read the WARNING lines first.

In [ ]:
# --- 4. Replay check on the CPU and the D0 report --------------------------------------------------------------------------------------
cpu = {**ENV, "JAX_PLATFORMS": "cpu"}
for split in ("train", "val", "test"):
    path = DATA / f"craftax_expert_{split}.h5"
    if path.exists():
        print(f"===== replay {split}")
        sh([PYTHON, REPO / "purejaxql" / "scripts" / "check_replay.py", "--data", path, "--num-episodes", REPLAY_EPISODES], env=cpu, cwd=REPO / "purejaxql", check=False)

sh([*RUNNER, "run", "D0", "--spec", SPEC, "--mode", "local", "--force"], quiet=True)
report = (DATA / "reports" / "dataset_stats" / "dataset_stats.md").read_text()
print("")
for line in report.splitlines():
    if line.startswith(("## ", "All 7 entities", "Without the daylight", "Check:", "Reading of the gap")) or "WARNING" in line:
        print(line[:300])

In [ ]:
# --- 4. Upload the data and the D0 report to the same repo ---------------------------------------------------------------------------------
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA), path_in_repo="data", allow_patterns=["craftax_expert_*.h5"])
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA / "reports"), path_in_repo="reports")
print(f"done: https://huggingface.co/datasets/{HF_REPO_ID}")